In [ ]:
! pip install --upgrade --quiet  pinecone-client pinecone-text pinecone-notebooks! pip install --upgrade --quiet  pinecone-client pinecone-text pinecone-notebooks


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
! pip uninstall pinecone-client -y

Found existing installation: pinecone-client 6.0.0
Uninstalling pinecone-client-6.0.0:
  Successfully uninstalled pinecone-client-6.0.0


In [5]:
! pip install pinecone pinecone-text

   ---------------------------------------- 0.0/742.8 kB ? eta -:--:--
   ---------------------------- ----------- 524.3/742.8 kB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 742.8/742.8 kB 2.1 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

from pinecone import Pinecone,ServerlessSpec
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from pinecone_text.sparse import BM25Encoder
from langchain_community.retrievers import PineconeHybridSearchRetriever

In [8]:
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
api_key = os.getenv("PINECONE_API_KEY")

In [9]:
index_name="hybrid-search-langchain-pinecone"

## initialize the Pinecone client
pc=Pinecone(api_key=api_key)

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,  # dimensionality of dense model
        metric="dotproduct",  # sparse values supported only for dotproduct
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"Index '{index_name}' created successfully.")
else:
    print(f"Index '{index_name}' already exists.")

Index 'hybrid-search-langchain-pinecone' created successfully.


In [10]:
index=pc.Index(index_name)
index

In [13]:
## vector embedding and sparse matrix
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1873.80it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [12]:
bm25_encoder=BM25Encoder().default()
bm25_encoder

In [14]:
sentences=[
    "In 2023, I visited Paris",
        "In 2022, I visited New York",
        "In 2021, I visited New Orleans",

]

## tfidf values on these sentence
bm25_encoder.fit(sentences)

## store the values to a json file
bm25_encoder.dump("bm25_values.json")

# load to your BM25Encoder object
bm25_encoder = BM25Encoder().load("bm25_values.json")

100%|██████████| 3/3 [00:00<00:00,  8.91it/s]


In [15]:
retriever=PineconeHybridSearchRetriever(embeddings=embeddings,sparse_encoder=bm25_encoder,index=index,alpha=0.7)
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x000001B41702CBC0>, index=<pinecone.db_data.index.Index object at 0x000001B3D69B1E20>, alpha=0.7)

In [16]:
# Corrected list format
retriever.add_texts(
    [
        "In 2023, I visited Paris",
        "In 2022, I visited New York",
        "In 2021, I visited New Orleans"
    ]
)

# Querying the retriever
result = retriever.invoke("What happened in 2023?")

# To see the answer, you can print the content of the first document returned
print(result[0].page_content)

100%|██████████| 1/1 [00:04<00:00,  4.00s/it]


In 2023, I visited Paris


In [ ]:
# This will rely more on Semantic search because it's a general concept.
retriever.invoke("Tell me about my trips")

[Document(metadata={'score': 0.305463314}, page_content='In 2023, I visited Paris'),
 Document(metadata={'score': 0.330878735}, page_content='In 2022, I visited New York'),
 Document(metadata={'score': 0.299555779}, page_content='In 2021, I visited New Orleans')]

In [18]:
# This will rely heavily on Keyword search because it's a specific "sparse" token.
retriever.invoke("2021") 

[Document(metadata={'score': 0.462623209}, page_content='In 2021, I visited New Orleans'),
 Document(metadata={'score': 0.325660229}, page_content='In 2022, I visited New York'),
 Document(metadata={'score': 0.310526848}, page_content='In 2023, I visited Paris')]

In [19]:
# 1. The "Trips" Query (Semantic Dominance)

# Observation: The scores are all very close (around 0.30 – 0.33).

# Why: Since the word "trips" doesn't actually appear in your sentences,
# the BM25 (Keyword) part of the engine found zero exact matches. 
# The retriever had to rely almost entirely on the Semantic (Dense) 
# side to realize that "visited Paris" is conceptually similar to a "trip."

# 2. The "2021" Query (Keyword Dominance)

# Observation: The score for the 2021 document jumped up to 0.46, while the others stayed lower.

# Why: This is the "Hybrid" power. The Semantic side saw them as similar, 
# but the Sparse (BM25) side found an exact match for the token "2021". 
# That extra "keyword boost" pushed that specific document far ahead of the others.

In [21]:
from langchain_classic.chains import RetrievalQA
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# 1. Initialize your NVIDIA NIM "Brain"
llm = ChatNVIDIA(model="nvidia/nemotron-3-super-120b-a12b", nvidia_api_key=os.getenv("NVIDIA_API_KEY"))

# 2. Create the RAG Chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", # "Stuff" means: take the docs and stuff them into the prompt
    retriever=retriever
)

# 3. Ask a question!
response = qa_chain.invoke("Which city did I visit most recently, and what was the year?")
print(response["result"])

Paris in 2023.


In [22]:
# 3. Ask a question!
response = qa_chain.invoke("Which city did I visit in 2021?")
print(response["result"])

New Orleans.
